### Step 2.1 Create Use Case Approval Tables

Master list of the BMI verified source variables.

Which approved input lists do we need?
|Input list	|Recommendation	|Why|
| --- | --- | ---- |
|approved_variables	|Yes — start here	Records each verified variable, its meaning, and Codebook citation.|
|approved_wave_coverage	|Yes, after variable verification	Records which variables apply to which waves and the supporting evidence.|
|approved_cohort_coverage	|Only if needed	Records cohort availability or applicability. Cohort membership may be better established from respondent data and documented cohort definitions.|
|approved_statistics	|Not as an input approval list	Calculate BMI means and other statistics from the validated survey data. Use Codebook descriptive statistics as a validation reference when appropriate.|
|approved_value_labels	|If needed	Captures documented codes and labels for categorical variables such as race and gender/sex.|
|approved_chunks	|Useful for the AI Agent, but separate	Preserves searchable Codebook passages for retrieval and traceability. It is not a list of approved analytical variables.|

### Create the Approved Variable Table

In [0]:
# ------------------------------------------------------------
# 5. Create the persistent Delta table if it does not exist
# ------------------------------------------------------------

approved_variable_table = ("dev_catalog.hrs_agent_metadata.hrs_use_case_approved_variables")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {approved_variable_table} (
        use_case_name STRING  COMMENT 'Identifies the use case',
        codebook_version STRING COMMENT 'Identifies the Codebook version',

        -- Resolved variable identity
        variable_name STRING  COMMENT 'For invariant variables such as RARACEM',
        variable_family STRING COMMENT 'Groups related variables, such as BMI',
        
        -- Selection request
        -- variable_pattern STRING comment 'Naming pattern like R{{wave}}BMI',
        -- variable_scope STRING  COMMENT 'INVARIANT OR WAVE_SPECIFIC',
        -- wave_start SMALLINT    COMMENT 'Starting wave for a wave-specific family',
        -- wave_end SMALLINT      COMMENT 'Ending wave for a wave-specific family',
        -- selection_type STRING COMMENT 'VARIABLE_FAMILY or SINGLE_VARIABLE',

        -- Source matching and approval
        bronze_column_match BOOLEAN COMMENT 'Indicates whether a source column was found',
        approval_status STRING COMMENT 'Tracks approval',

        -- Codebook documentation
        variable_label STRING COMMENT 'Stores Codebook documentation',
        definition STRING     COMMENT 'Stores Codebook documentation',
        survey_section STRING COMMENT 'Stores Codebook documentation',
        measurement_unit STRING  COMMENT 'Stores Codebook documentation',
        source_page INT  COMMENT 'Stores Codebook documentation',
        
        -- Review and audit
        reviewed_by STRING COMMENT 'Audit Comlmn: Track review and maintenance',
        reviewed_at TIMESTAMP  COMMENT 'Audit Comlmn: Track review and maintenance',
        created_at TIMESTAMP  COMMENT 'Audit Comlmn: Track review and maintenance',
        updated_at TIMESTAMP COMMENT 'Audit Comlmn: Track review and maintenance'
    )
    USING DELTA
""")

# Ensure columns exist in case the table was created by an earlier version
existing_columns = [f.name for f in spark.table(approved_variable_table).schema.fields]
for col_name, col_type in [("wave_number", "STRING"), ("wave_coverage_status", "STRING")]:
    if col_name not in existing_columns:
        spark.sql(f"ALTER TABLE {approved_variable_table} ADD COLUMNS ({col_name} {col_type})")

### Create Table HRS Use Case Approved Cohort Coverage
Master list of the verified source variables.

In [0]:
# ------------------------------------------------------------
# 1. Create the approved cohort coverage table
# ------------------------------------------------------------

coverage_table = (
    "dev_catalog.hrs_agent_metadata."
    "hrs_use_case_approved_cohort_coverage"
)

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {coverage_table} (
    -- Identity Columns
    use_case_name STRING,
    codebook_version STRING,
    variable_name STRING,
    -- Codebook Documentation
    wave_number INT,
    cohort_number DOUBLE,
    non_null_record_count BIGINT,
    -- Status
    coverage_status STRING,
    -- Review and Audit columns
    reviewed_by STRING,
    reviewed_at TIMESTAMP,
    created_at TIMESTAMP,
    updated_at TIMESTAMP
)
USING DELTA
""")

print(f"Coverage table ready: {coverage_table}")